# QLoRA Domain Specialist – Exploration Notebook

This notebook is a companion to the main training scripts.
Use it to:
- Inspect the dataset
- Experiment with LoRA rank / alpha
- Quickly test generation before full training

**Recommended environment:** Google Colab (T4) or any GPU with ≥ 8 GB VRAM.

In [ ]:
# Install (uncomment if needed)
# !pip install -q transformers peft trl bitsandbytes accelerate datasets gradio

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Load model in 4-bit (QLoRA style)

In [ ]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print("Model loaded in 4-bit.")

## 2. Attach LoRA adapters and check trainable parameters

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 3. Quick generation test

In [ ]:
prompt = "How do I reset my password?"
messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=128, do_sample=True, temperature=0.7)
print(tokenizer.decode(out[0], skip_special_tokens=True))

## Next steps

1. Run the full training script: `python train_qlora.py`
2. Evaluate: `python evaluate.py`
3. Launch demo: `python app.py`